# Silver Transformation - dimpromotable

This notebook builds the `dimpromotable` dataset in the Silver layer.



## Run Shared Libraries



In [ ]:
%run ../Misc/sharedlibraries

In [ ]:
UpdatedDateTime = datetime.datetime.now()
Entity = "dimpromotable"

## Read Source Tables



In [ ]:
promotablebronzedf = spark.table("bronze.promotable")

## Build Silver Dataset



In [ ]:
promotabledf = promotablebronzedf.filter(promotablebronzedf.RecordId.isNotNull()
    ).select(
        promotablebronzedf.PromotionId,
        F.when(promotablebronzedf.LastProcessedChange_DateTime.isNull(), F.lit("1900-01-01")).otherwise(promotablebronzedf.LastProcessedChange_DateTime).cast("timestamp").alias("LastProcessedChange_DateTime"),
        F.from_utc_timestamp(promotablebronzedf.DataLakeModified_DateTime,'CST').alias("DataLakeModified_DateTime"),
        F.trim(promotablebronzedf.PromotionName).alias("PromotionName"),
        promotablebronzedf.PromoCode,
        F.trim(promotablebronzedf.PromoType).alias("PromoType"),
        promotablebronzedf.PromoPercentage,
        F.from_utc_timestamp(promotablebronzedf.ValidFrom,'CST').alias("ValidFrom"),
        F.when(promotablebronzedf.ValidTo.isNull(), F.lit("1900-01-01")).otherwise(promotablebronzedf.ValidTo).cast("timestamp").alias("ValidTo"),
        promotablebronzedf.IsActive,
        promotablebronzedf.RecordId.alias("PromoRecordId")
    ).withColumn("UpdatedDateTime", F.lit(UpdatedDateTime)
    ).withColumn("PromoHashKey", F.xxhash64("PromoRecordId")
    )

display(promotabledf)

## Prepare Final DataFrame



In [ ]:
df_final = promotabledf

## Verify Source Schema



In [ ]:
saveDeltaTableToCatalog(df_final,"silver",Entity)